# Lesson 7.1 — Multi-Account Strategy: AWS Organizations, Control Tower & RAM

**Level: Advanced** | **Exam:** SAP-C02 Domain 1, Task 1.4

## Multi-Account AWS Strategy

### Why Multiple Accounts?
- **Security isolation**: Blast radius containment
- **Cost allocation**: Each account = natural billing boundary  
- **Service limits**: Limits are per account
- **Compliance**: Separate workloads with different compliance requirements

### Account Structure (Landing Zone)
```
Management Account (root)
├── Security OU
│   ├── Log Archive Account (CloudTrail, Config logs)
│   └── Security Tooling Account (GuardDuty, Security Hub)
├── Infrastructure OU
│   └── Network Account (Transit Gateway, Shared VPC)
└── Workloads OU
    ├── Dev / Staging / Prod accounts
    └── Sandbox OU
```

### AWS Organizations
- Root → Organizational Units (OUs) → Accounts
- **SCPs** applied at Root, OU, or Account level
- **Consolidated billing**: volume discounts, RI sharing

### AWS Control Tower
- Automates multi-account setup with best practices
- Deploys Landing Zone with Log Archive + Security Tooling accounts  
- **Guardrails** = pre-configured SCPs and Config rules
- **Account Factory**: Self-service account vending machine

### AWS Resource Access Manager (RAM)
Share AWS resources across accounts WITHOUT creating peering:
- Transit Gateway attachments, Subnets, Route 53 Resolver rules

In [ ]:
import boto3, json
account_id = boto3.client('sts').get_caller_identity()['Account']
print(f'Account: {account_id}')

In [ ]:
# Explore AWS Organizations structure
import boto3

org = boto3.client('organizations', region_name='us-east-1')

try:
    org_info = org.describe_organization()['Organization']
    print(f"Organization ID: {org_info['Id']}")
    print(f"Master Account: {org_info['MasterAccountId']}")
    print(f"Feature set: {org_info['FeatureSet']}")
    
    paginator = org.get_paginator('list_accounts')
    accounts = []
    for page in paginator.paginate():
        accounts.extend(page['Accounts'])
    
    print(f"\nAccounts ({len(accounts)}):")
    for acct in accounts:
        print(f"  {acct['Id']}  {acct['Name']:40}  {acct['Status']}")
        
    root_id = org.list_roots()['Roots'][0]['Id']
    ous = org.list_organizational_units_for_parent(ParentId=root_id)['OrganizationalUnits']
    print(f"\nTop-level OUs ({len(ous)}):")
    for ou in ous:
        print(f"  {ou['Id']}  {ou['Name']}")
        
except org.exceptions.AWSOrganizationsNotInUseException:
    print("This account is not part of an AWS Organization.")
    print("To create: org.create_organization(FeatureSet='ALL')")
except Exception as e:
    print(f"Organizations: {e}")

## Exercise

In [ ]:
# TODO: Write a function that traverses the entire AWS Organizations hierarchy
# and outputs a tree: Root -> OUs -> Accounts (with indentation showing depth)
# Hint: use list_organizational_units_for_parent() and list_accounts_for_parent()

import boto3

def print_org_tree(org_client, parent_id, indent=0):
    prefix = "  " * indent
    # Get OUs at this level
    ous = org_client.list_organizational_units_for_parent(ParentId=parent_id)['OrganizationalUnits']
    for ou in ous:
        print(f"{prefix}OU: {ou['Name']} ({ou['Id']})")
        # TODO: Recurse into this OU
        # TODO: List accounts in this OU
    
    # TODO: Get and print accounts at this level

try:
    org = boto3.client('organizations', region_name='us-east-1')
    root = org.list_roots()['Roots'][0]
    print(f"Root: {root['Id']}")
    print_org_tree(org, root['Id'])
except Exception as e:
    print(f"Error: {e}")

## Summary
- Multiple accounts = security isolation, cost allocation, service limit separation
- **AWS Organizations**: Hierarchical account management with SCPs as guardrails
- **Control Tower**: Automated Landing Zone with best-practice account vending
- **RAM**: Share resources (TGW, subnets) across accounts without peering
- Always use consolidated billing — enables RI sharing and volume discounts

---
**Next:** [Lesson 7.2 — Advanced Networking: Direct Connect & VPN](lesson-7.2-hybrid-networking.ipynb)